In [136]:
import pickle as pkl
model=pkl.dump(model,open('model.pkl','wb'))
Transformer=pkl.dump(Transformer,open("Transformer.pkl","wb"))
colText=pkl.dump(colText,open('coltext','wb'))
colNum=pkl.dump(colNum,open('colNum','))

In [50]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score,precision_score,accuracy_score,confusion_matrix,classification_report,recall_score
# from sklearn.
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer

In [2]:
fraud=pd.read_csv("fraudDetection.csv")

In [3]:
columnsToExclude=['customer','gender','zipcodeOri','zipMerchant','category']
fraud=fraud.drop(columns=columnsToExclude)

In [4]:
fraud=fraud.drop(columns=['step'])

In [5]:
# Remove double qoute
for qoute in fraud.select_dtypes(include="object"):
    fraud[qoute]=fraud[qoute].str.replace("'","",regex=False).str.strip()

In [6]:
fraud['age']=pd.to_numeric(fraud['age'],errors="coerce")

In [7]:
fraud['age']=fraud['age'].fillna(fraud['age'].median())

In [8]:
fraud[fraud['fraud']==1]

,age,merchant,amount,fraud
88,3.0,M480139044,44.26,1
89,3.0,M480139044,324.50,1
434,3.0,M857378720,176.32,1
435,3.0,M857378720,337.41,1
553,4.0,M1198415165,220.11,1
...,...,...,...,...
593928,5.0,M3697346,192.78,1
594025,5.0,M1748431652,42.37,1
594026,3.0,M1748431652,521.84,1
594168,2.0,M209847108,25.29,1


In [9]:
fraud

,age,merchant,amount,fraud
0,4.0,M348934600,4.55,0
1,2.0,M348934600,39.68,0
2,4.0,M1823072687,26.89,0
3,3.0,M348934600,17.25,0
4,5.0,M348934600,35.72,0
...,...,...,...,...
594638,3.0,M1823072687,20.53,0
594639,4.0,M1823072687,50.73,0
594640,2.0,M349281107,22.44,0
594641,5.0,M1823072687,14.46,0


In [10]:
# remove Qoute and white space
# fraud['category']=fraud['category'].str.replace("es_","").str.strip()

In [11]:
fraud[fraud['fraud']==1]

,age,merchant,amount,fraud
88,3.0,M480139044,44.26,1
89,3.0,M480139044,324.50,1
434,3.0,M857378720,176.32,1
435,3.0,M857378720,337.41,1
553,4.0,M1198415165,220.11,1
...,...,...,...,...
593928,5.0,M3697346,192.78,1
594025,5.0,M1748431652,42.37,1
594026,3.0,M1748431652,521.84,1
594168,2.0,M209847108,25.29,1


In [12]:
# check null values
fraud.isnull().sum()

age         0
merchant    0
amount      0
fraud       0
dtype: int64

In [13]:
fraud

,age,merchant,amount,fraud
0,4.0,M348934600,4.55,0
1,2.0,M348934600,39.68,0
2,4.0,M1823072687,26.89,0
3,3.0,M348934600,17.25,0
4,5.0,M348934600,35.72,0
...,...,...,...,...
594638,3.0,M1823072687,20.53,0
594639,4.0,M1823072687,50.73,0
594640,2.0,M349281107,22.44,0
594641,5.0,M1823072687,14.46,0


In [14]:
# # chek outliers
# for outliers in fraud.columns:
#     if fraud[outliers].dtype!="object":
#         plt.figure(figsize=(20,6))
#         sns.boxplot(x=fraud[outliers])

In [15]:
# fraud['age']=fraud['age'].astype(int)

In [16]:
fraud['fraud'].unique()

array([0, 1])

In [17]:
# handle outliers
for handleOut in fraud.columns:
    if fraud[handleOut].dtype!="object" and handleOut!="fraud":
        Q1=fraud[handleOut].quantile(0.25)
        Q3=fraud[handleOut].quantile(0.75)
        IQR=Q3-Q1

        Upper=Q3+1.5*IQR
        Lower=Q1-1.5*IQR

        

        count=fraud[(fraud[handleOut]<Lower) | (fraud[handleOut]>Upper)].shape[0]
        if count!=0:
            fraud[handleOut]=fraud[handleOut].clip(lower=Lower,upper=Upper)

In [18]:
# check skewness
for skew in fraud.columns :
    if fraud[skew].dtype!="object":
        skewness=fraud[skew].skew()
        print(f"{skew}  {skewness}")

age  0.43110904693741
amount  0.8468415947963844
fraud  8.921993255938798


In [19]:
fraud[fraud['fraud']==1]

,age,merchant,amount,fraud
88,3.0,M480139044,44.26,1
89,3.0,M480139044,85.74,1
434,3.0,M857378720,85.74,1
435,3.0,M857378720,85.74,1
553,4.0,M1198415165,85.74,1
...,...,...,...,...
593928,5.0,M3697346,85.74,1
594025,5.0,M1748431652,42.37,1
594026,3.0,M1748431652,85.74,1
594168,2.0,M209847108,25.29,1


In [20]:
fraud

,age,merchant,amount,fraud
0,4.0,M348934600,4.55,0
1,2.0,M348934600,39.68,0
2,4.0,M1823072687,26.89,0
3,3.0,M348934600,17.25,0
4,5.0,M348934600,35.72,0
...,...,...,...,...
594638,3.0,M1823072687,20.53,0
594639,4.0,M1823072687,50.73,0
594640,2.0,M349281107,22.44,0
594641,5.0,M1823072687,14.46,0


In [21]:
fraud["amount"].corr(fraud['fraud'],method="pearson")

np.float64(0.2537194415299726)

In [22]:
sns.heatmap(fraud[colNumber].corr(),annot=True)

NameError: name 'colNumber' is not defined

In [ ]:
# encode .
xTrain=Transformer.fit_transform(xTrain)

In [ ]:
#Box Plot
for box in fraud.columns:
    if fraud[box].dtype!="object":
        plt.figure(figsize=(20,5))
        sns.boxplot(x=fraud['fraud'],y=fraud[box])

In [27]:
# Check corr
colNumber=[cols for cols in fraud.columns if fraud[cols].dtype!="object"]
colNumber
colText=[coltext for coltext in fraud.columns if fraud[coltext].dtype=="object"]
colText

['merchant']

In [28]:
# Feature encoding
pipelineText=Pipeline(

    [("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]
)
Transformer=ColumnTransformer(
    [("encoding",pipelineText,colText)]
)


In [ ]:
x

In [29]:
# split data
x=fraud.drop(columns=['fraud'])
y=fraud['fraud']
xTrainTempo,xTest,yTrainTempo,yTest=train_test_split(x,y,test_size=0.2,random_state=34)
xTrain,xTestVal,yTrain,yTestVal=train_test_split(xTrainTempo,yTrainTempo,test_size=0.2, random_state=34)

In [30]:
# Encode Training set
xTrain=Transformer.fit_transform(xTrain)

In [31]:
# Encode xTest 
xTest=Transformer.transform(xTest)

In [32]:
# encode xtestVal
xTestVal=Transformer.transform(xTestVal)


In [ ]:
# xTestVal

In [33]:
# display xTrain with features
xTrain=pd.DataFrame(xTrain,columns=Transformer.get_feature_names_out())


In [35]:
# display xTest with features
xTest=pd.DataFrame(xTest,columns=Transformer.get_feature_names_out())


In [36]:
# display xTestVal with features
xTestVal=pd.DataFrame(xTestVal,columns=Transformer.get_feature_names_out())


In [ ]:
# xTrain

In [116]:
model = XGBClassifier(
    n_estimators          = 400,
    learning_rate         = 0.1,
    max_depth             = 3,
    reg_alpha             = 0.1,
    reg_lambda            = 0.1,
    scale_pos_weight      = 5.0,  
    eval_metric           = "logloss",
    early_stopping_rounds = 50
)

model.fit(
    xTrain, yTrain,
    eval_set = [(xTestVal, yTestVal)],
    verbose  = False
)

prediction  = model.predict(xTest)
recallScore = recall_score(yTest, prediction)
print(f"Recall: {recallScore:.4f}")

Recall: 0.8245


In [117]:
reportFrame.transpose()

,precision,recall,f1-score,support
No Fraud,0.997754,0.990940,0.994335,117436.00000
Fraud,0.536383,0.824514,0.649947,1493.00000
accuracy,0.988850,0.988850,0.988850,0.98885
macro avg,0.767069,0.907727,0.822141,118929.00000
weighted avg,0.991962,0.988850,0.990012,118929.00000


In [118]:
# preddiction
predict=model.predict(xTest)

In [67]:
# fraud['fraud'].value_counts()

In [119]:
predict

array([0, 0, 0, ..., 0, 0, 0], shape=(118929,))

In [120]:
probability=model.predict_proba(xTrain)

In [121]:
probaFrame=pd.DataFrame(probability,columns=model.classes_)

In [122]:
probaFrame.columns=['No Fraud','Fraud']

In [123]:
probaFrame

,No Fraud,Fraud
0,0.999963,0.000037
1,0.999963,0.000037
2,0.999963,0.000037
3,0.999963,0.000037
4,0.999963,0.000037
...,...,...
380566,0.999554,0.000446
380567,0.999963,0.000037
380568,0.999963,0.000037
380569,0.999945,0.000055


In [124]:
# extract classification report
report=classification_report(yTest,predict,target_names=model.classes_,output_dict=True)

In [125]:
report

{np.int64(0): {'precision': 0.9977536567381724,
  'recall': 0.990939745904152,
  'f1-score': 0.9943350279830819,
  'support': 117436.0},
 np.int64(1): {'precision': 0.5363834422657952,
  'recall': 0.8245144005358339,
  'f1-score': 0.6499472016895459,
  'support': 1493.0},
 'accuracy': 0.9888504906288625,
 'macro avg': {'precision': 0.7670685495019838,
  'recall': 0.9077270732199929,
  'f1-score': 0.8221411148363139,
  'support': 118929.0},
 'weighted avg': {'precision': 0.9919617495481072,
  'recall': 0.9888504906288625,
  'f1-score': 0.9900116835956217,
  'support': 118929.0}}

In [126]:
reportFrame=pd.DataFrame(report)

In [127]:
# classification report frame
reportFrame.columns=["No Fraud", "Fraud", 'accuracy', 'macro avg', 'weighted avg']

In [128]:
reportFrame.transpose()

,precision,recall,f1-score,support
No Fraud,0.997754,0.990940,0.994335,117436.00000
Fraud,0.536383,0.824514,0.649947,1493.00000
accuracy,0.988850,0.988850,0.988850,0.98885
macro avg,0.767069,0.907727,0.822141,118929.00000
weighted avg,0.991962,0.988850,0.990012,118929.00000
